In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import numpy as np
from sklearn import metrics

from sklearn.metrics import precision_score, roc_auc_score


In [2]:
main_unique = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/new_analysis_after_new_agg_21082025/input_data/3_ML_outputs/datasets/PredictingSLResistanceFeatures_main_withfeatures_dropna_remove_duplicate.csv')
main_unique

/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_25468/3781834036.py:1: DtypeWarning: Columns (5,16,17) have mixed types. Specify dtype option on import or set low_memory=False.
  main_unique = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/new_analysis_after_new_agg_21082025/input_data/3_ML_outputs/datasets/PredictingSLResistanceFeatures_main_withfeatures_dropna_remove_duplicate.csv')


,Screen,SL_Pair,Query,Biomarker,Target1,Target2,entrez_id_query,hgnc_id_query,ensembl_gene_id_query,entrez_id_biomarker,...,FET_SharedInteractors_Biomarker_STRING,FET_SharedInteractors_Target_STRING,ExpressionVariance,EssentialityVariance,EssentialityAverage,BIOGRIDPhysicalInteractionQueryBiomarker,BIOGRIDPhysicalInteractionQueryTarget,BiomarkerType,Essentiality_Percentage,Class_Processed
0,Dunn,PTEN_PIK3CB,PIK3R2,PTEN,PIK3CB,NaN,5296.0,HGNC:8980,ENSG00000105647,5728.0,...,114.152228,323.306215,291.159967,0.019359,0.131894,1,1,1,0.509338,1
1,Dunn,PTEN_PIK3CB,INPPL1,PTEN,PIK3CB,NaN,3636.0,HGNC:6080,ENSG00000165458,5728.0,...,34.068956,83.017283,1666.329823,0.019238,-0.020458,0,0,1,0.000000,1
2,Dunn,PTEN_PIK3CB,TSC2,PTEN,PIK3CB,NaN,7249.0,HGNC:12363,ENSG00000103197,5728.0,...,144.356054,62.190596,533.850119,0.194427,0.089340,0,0,1,11.629881,1
3,Dunn,PTEN_PIK3CB,NF2,PTEN,PIK3CB,NaN,4771.0,HGNC:7773,ENSG00000186575,5728.0,...,68.669701,34.650616,46.175986,0.155817,-0.340644,0,0,1,1.103565,1
4,Dunn,PTEN_PIK3CB,CXorf56,PTEN,PIK3CB,NaN,63932.0,HGNC:26239,ENSG00000018610,5728.0,...,-0.097172,-0.188936,45.769746,0.074343,0.205940,0,0,1,8.064516,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
142977,Hayes_Krall,NRAS_MEK,ZYG11A,NRAS,MEK1,MEK2,440590.0,HGNC:32058,ENSG00000203995,4893.0,...,0.000000,0.000000,2.097155,0.009856,0.053250,0,0,0,0.000000,0
142978,Hayes_Krall,NRAS_MEK,ZYG11B,NRAS,MEK1,MEK2,79699.0,HGNC:25820,ENSG00000162378,4893.0,...,-0.202768,0.000000,102.933050,0.016370,0.145534,0,0,0,0.424448,0
142979,Hayes_Krall,NRAS_MEK,ZYX,NRAS,MEK1,MEK2,7791.0,HGNC:13200,ENSG00000159840,4893.0,...,9.351460,16.005291,73339.950606,0.016416,0.040647,0,0,0,0.000000,0
142980,Hayes_Krall,NRAS_MEK,ZZEF1,NRAS,MEK1,MEK2,23140.0,HGNC:29027,ENSG00000074755,4893.0,...,-0.203459,0.000000,102.379095,0.014429,0.132027,0,0,0,0.084890,0


In [4]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier

# === Load training dataset (with labels) ===
main_unique = pd.read_csv(
    "/Users/metinyazar/Desktop/drug_resistance_ppi/new_analysis_after_new_agg_21082025/input_data/3_ML_outputs/datasets/PredictingSLResistanceFeatures_main_withfeatures_dropna_remove_duplicate.csv"
)

# Training features + labels
X_train = main_unique.iloc[:, -19:-1].values
y_train = main_unique["Class_Processed"].values

# === Train Random Forest ===
rf = RandomForestClassifier(
    n_estimators=1000,
    max_features=1,
    max_depth=15,
    min_samples_leaf=4,
    random_state=42
)
rf.fit(X_train, y_train)

# === Function: apply trained model to SL_pair dataset ===
def predict_sl_pair_proba(sl_pair_file: str, output_file: str):
    """
    Apply trained RandomForest model to SL_pair dataset and save probabilities.
    
    Parameters
    ----------
    sl_pair_file : str
        Path to the SL_pair dataset with features (no labels).
    output_file : str
        Path to save CSV with predicted probabilities.
    """
    sl_pair = pd.read_csv(sl_pair_file)
    
    # Extract the same feature columns used in training
    X_sl = sl_pair.iloc[:, -18:].values
    
    # Predict probabilities
    sl_pair["Resistance_Score"] = rf.predict_proba(X_sl)[:, 1]
    
    # Save results
    sl_pair.to_csv(output_file, index=False)
    print(f"✅ Predictions saved → {output_file}")
    return sl_pair

# === Example: Apply to ATM_ATR dataset ===
sl_pair_file = "datasets/PredictingSLResistanceFeatures_MTAP_PRMT5_main_withfeatures.csv"
output_file = "predictions/PredictingSLResistanceFeatures_MTAP_PRMT5_main_withfeatures_pred.csv"

predicted_sl_pair = predict_sl_pair_proba(sl_pair_file, output_file)


/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_25468/2082976980.py:6: DtypeWarning: Columns (5,16,17) have mixed types. Specify dtype option on import or set low_memory=False.
  main_unique = pd.read_csv(


✅ Predictions saved → predictions/PredictingSLResistanceFeatures_MTAP_PRMT5_main_withfeatures_pred.csv


In [5]:
import pandas as pd
import glob
import os

# Directory containing prediction CSVs
input_dir = "predictions"
output_file = "predictions/supplementary_file_clinical_trial_predictions.xlsx"

# Find all prediction CSVs
csv_files = glob.glob(os.path.join(input_dir, "PredictingSLResistanceFeatures_*_main_withfeatures_pred.csv"))

# Write to one Excel file with multiple sheets
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    for file in csv_files:
        # Extract SL_pair from filename
        # Filename pattern: PredictingSLResistanceFeatures_{SL_pair}_main_withfeatures_pred.csv
        basename = os.path.basename(file)
        sl_pair = basename.replace("PredictingSLResistanceFeatures_", "").replace("_main_withfeatures_pred.csv", "")

        # Load CSV
        df = pd.read_csv(file)

        # Write each SL_pair to its own sheet
        df.to_excel(writer, sheet_name=sl_pair[:31], index=False)  
        # Excel sheet names max length = 31 chars → truncate if needed

print(f"✅ Combined predictions saved in {output_file}")


✅ Combined predictions saved in predictions/supplementary_file_clinical_trial_predictions.xlsx
